# T2.3 · Rolling 30-day evaluation

Metrics: **Brier** on `P(outage)`, **MAE** on `duration_min` where `outage==1`, and **lead time** to true outage starts (hours from first crossing of an alert threshold in the prior 24h).

In [ ]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecaster import Forecaster, load_grid_csv
from prioritizer import load_plan_inputs, plan

In [ ]:
df = load_grid_csv("grid_history.csv")
n = len(df)
ROLL_DAYS = 30
ALERT_P = 0.35

origins = [n - 25 - 24 * d for d in range(ROLL_DAYS)]
origins = [o for o in origins if o >= 220]
print("origins:", len(origins), "first", origins[-1], "last", origins[0])

In [ ]:
brier_terms: list[float] = []
dur_abs: list[float] = []
lead_samples: list[float] = []
rows_flat: list[dict] = []

for o in origins:
    train = df.iloc[: o + 1].copy()
    m = Forecaster(min_train_rows=300)
    m.fit(train, max_label_index=o)
    fc = m.predict_next_24h(df.iloc[: o + 25], origin_idx=o)
    for h in range(24):
        j = o + 1 + h
        y = int(df.at[j, "outage"])
        p = float(fc[h]["p_outage"])
        brier_terms.append((p - y) ** 2)
        if y == 1:
            pred_d = float(fc[h]["e_duration_min"])
            act_d = float(df.at[j, "duration_min"])
            dur_abs.append(abs(pred_d - act_d))
        rows_flat.append({"origin": o, "h": h, "j": j, "p": p, "y": y})

    # Lead time for outage starts in this 24h window
    for h in range(24):
        j = o + 1 + h
        prev = int(df.at[j - 1, "outage"]) if j > 0 else 0
        cur = int(df.at[j, "outage"])
        if cur == 1 and prev == 0:
            past_ps = [float(fc[k]["p_outage"]) for k in range(h)]
            fired = [k for k, pk in enumerate(past_ps) if pk >= ALERT_P]
            if fired:
                lead_samples.append(h - fired[0])
            else:
                lead_samples.append(0.0)

brier = float(np.mean(brier_terms)) if brier_terms else float("nan")
mae_dur = float(np.mean(dur_abs)) if dur_abs else float("nan")
lead_mean = float(np.mean(lead_samples)) if lead_samples else float("nan")

print(f"Brier (P outage): {brier:.5f}")
print(f"MAE duration (outage hours only): {mae_dur:.2f} min  (n={len(dur_abs)})")
print(f"Mean lead time (starts only, alert p>={ALERT_P}): {lead_mean:.2f} h  (events={len(lead_samples)})")

In [ ]:
# Worst single forecast hour in the held-out rolling window (highest Brier contribution)
worst = max(rows_flat, key=lambda r: (r["p"] - r["y"]) ** 2)
row = df.iloc[worst["j"]]
print("Worst hour (by squared error):")
print("  idx", worst["j"], "UTC", row["timestamp"], "p", round(worst["p"], 4), "y", worst["y"])
print("  load_mw", round(float(row["load_mw"]), 2), "rain_mm", round(float(row["rain_mm"]), 2), "hour", pd.Timestamp(row["timestamp"]).hour)

In [ ]:
# Quick plan check (salon) on last origin
o = origins[0]
m = Forecaster(min_train_rows=300)
m.fit(df.iloc[: o + 1], max_label_index=o)
fc = m.predict_next_24h(df.iloc[: o + 25], origin_idx=o)
biz = next(b for b in json.load(open("businesses.json")) if b["id"] == "salon")
apps, genw = load_plan_inputs(biz)
sched = plan(fc, apps, genw)
print("shed_order:", " -> ".join(sched["shed_order"]))
print("sample hour0:", sched["hours"][0]["appliances"])

In [ ]:
plt.figure(figsize=(9, 3.2))
plt.title("Rolling-window Brier contributions (all forecast hours)")
plt.plot(np.asarray(brier_terms), lw=1)
plt.ylabel("(p - y)^2")
plt.xlabel("forecast hour index")
plt.tight_layout()
plt.show()